# 🗄️ SQL Mastery Notebook — Basic to Advanced

Master SQL from scratch to expert level using **PySpark SQL** (pure SQL syntax via `spark.sql()`).

## 📚 Topics Covered:
| # | Topic | Difficulty |
|---|-------|-----------|
| 1 | Setup & Sample Data | 🔧 Setup |
| 2 | SELECT, WHERE, ORDER BY, LIMIT | 🟢 Basic |
| 3 | DISTINCT, ALIAS, LIKE, IN, BETWEEN | 🟢 Basic |
| 4 | Aggregations — COUNT, SUM, AVG, MIN, MAX | 🟡 Medium |
| 5 | GROUP BY + HAVING | 🟡 Medium |
| 6 | JOINs — INNER, LEFT, RIGHT, FULL OUTER | 🟡 Medium |
| 7 | Subqueries & CTEs (WITH clause) | 🔴 Advanced |
| 8 | Window Functions — ROW_NUMBER, RANK, DENSE_RANK | 🔴 Advanced |
| 9 | Window Functions — LAG, LEAD, NTILE, Running Total | 🔴 Advanced |
| 10 | CASE WHEN, COALESCE, NULL Handling | 🔴 Advanced |
| 11 | String & Date Functions | 🔴 Advanced |
| 12 | Interview-Style Practice Problems | 🏆 Challenge |

---
> **How to use**: Run cells top to bottom. Each section is self-contained with explanations + runnable queries.

---
## 🔧 Section 1 — Setup: SparkSession
Start Spark and configure Java. Run this cell first — everything else depends on it.

In [1]:
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["PATH"] = f"/opt/homebrew/opt/openjdk@17/bin:{os.environ['PATH']}"

from pyspark.sql import SparkSession

# Stop existing session (safe re-run)
try:
    SparkSession.getActiveSession().stop()
except Exception:
    pass

spark = SparkSession.builder \
    .appName("SQL Mastery") \
    .master("local[*]") \
    .config("spark.ui.port", "4055") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"✅ Spark {spark.version} ready!")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/01 19:08:27 WARN Utils: Your hostname, Mohits-MacBook-Pro-2.local, resolves to a loopback address: 127.0.0.1; using 192.168.4.168 instead (on interface en0)
26/03/01 19:08:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/01 19:08:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Spark 4.1.1 ready!


---
## 🗂️ Section 2 — Sample Data Setup

We create three tables used throughout the entire notebook:

| Table | Description |
|-------|-------------|
| `employees` | Employee records with salary, department, hire date |
| `departments` | Department info with budget and location |
| `transactions` | Sales transactions linked to employees |

In [3]:
from pyspark.sql import Row
from datetime import date

# ── employees ─────────────────────────────────────────────────────────────────
employees_data = [
    (1,  "Alice",    "Engineering",  95000, "F", "2019-03-15", 3, 45),
    (2,  "Bob",      "Engineering",  88000, "M", "2020-07-01", 3, 32),
    (3,  "Carol",    "Marketing",    72000, "F", "2018-11-20", 2, 38),
    (4,  "David",    "Marketing",    68000, "M", "2021-01-10", 2, 29),
    (5,  "Eve",      "HR",           62000, "F", "2017-05-05", 1, 41),
    (6,  "Frank",    "HR",           59000, "M", "2022-09-12", 1, 27),
    (7,  "Grace",    "Engineering",  102000,"F", "2016-02-28", 3, 50),
    (8,  "Hank",     "Finance",      85000, "M", "2020-04-17", 4, 34),
    (9,  "Ivy",      "Finance",      91000, "F", "2019-08-22", 4, 36),
    (10, "Jack",     "Marketing",    74000, "M", "2018-06-30", 2, 42),
    (11, "Karen",    "Engineering",  None,  "F", "2023-01-05", 3, 24),
    (12, "Leo",      "Finance",      78000, "M", "2021-11-15", 4, 31),
]
emp_cols = ["emp_id","name","department","salary","gender","hire_date","dept_id","age"]
emp_df = spark.createDataFrame(employees_data, emp_cols)
emp_df.createOrReplaceTempView("employees")

# ── departments ───────────────────────────────────────────────────────────────
dept_data = [
    (1, "HR",          500000,  "New York"),
    (2, "Marketing",   800000,  "Chicago"),
    (3, "Engineering", 2000000, "San Francisco"),
    (4, "Finance",     1200000, "New York"),
    (5, "Legal",       600000,  "Boston"),      # no employees → tests LEFT JOIN
]
dept_cols = ["dept_id","dept_name","budget","location"]
dept_df = spark.createDataFrame(dept_data, dept_cols)
dept_df.createOrReplaceTempView("departments")

# ── transactions ──────────────────────────────────────────────────────────────
txn_data = [
    (101, 1,  2500.00, "2026-01-05", "Electronics"),
    (102, 1,  1800.00, "2026-01-20", "Electronics"),
    (103, 2,  3200.00, "2026-01-08", "Software"),
    (104, 3,   950.00, "2026-02-01", "Marketing"),
    (105, 3,  1100.00, "2026-02-14", "Marketing"),
    (106, 4,   750.00, "2026-01-30", "Marketing"),
    (107, 5,   400.00, "2026-02-10", "HR Tools"),
    (108, 7,  5500.00, "2026-01-15", "Cloud"),
    (109, 7,  4200.00, "2026-02-05", "Cloud"),
    (110, 8,  2100.00, "2026-01-22", "Finance"),
    (111, 9,  3800.00, "2026-02-18", "Finance"),
    (112, 10, 1600.00, "2026-01-11", "Marketing"),
]
txn_cols = ["txn_id","emp_id","amount","txn_date","category"]
txn_df = spark.createDataFrame(txn_data, txn_cols)
txn_df.createOrReplaceTempView("transactions")

print("✅ Tables registered: employees, departments, transactions")
print(f"   employees   : {emp_df.count()} rows")
print(f"   departments : {dept_df.count()} rows")
print(f"   transactions: {txn_df.count()} rows")

✅ Tables registered: employees, departments, transactions
   employees   : 12 rows
   departments : 5 rows
   transactions: 12 rows


---
## 🟢 Section 3 — Basic SQL: SELECT, WHERE, ORDER BY, LIMIT

The foundation of every SQL query.

In [4]:
# SELECT all columns
spark.sql("SELECT * FROM employees").show()


+------+-----+-----------+------+------+----------+-------+---+
|emp_id| name| department|salary|gender| hire_date|dept_id|age|
+------+-----+-----------+------+------+----------+-------+---+
|     1|Alice|Engineering| 95000|     F|2019-03-15|      3| 45|
|     2|  Bob|Engineering| 88000|     M|2020-07-01|      3| 32|
|     3|Carol|  Marketing| 72000|     F|2018-11-20|      2| 38|
|     4|David|  Marketing| 68000|     M|2021-01-10|      2| 29|
|     5|  Eve|         HR| 62000|     F|2017-05-05|      1| 41|
|     6|Frank|         HR| 59000|     M|2022-09-12|      1| 27|
|     7|Grace|Engineering|102000|     F|2016-02-28|      3| 50|
|     8| Hank|    Finance| 85000|     M|2020-04-17|      4| 34|
|     9|  Ivy|    Finance| 91000|     F|2019-08-22|      4| 36|
|    10| Jack|  Marketing| 74000|     M|2018-06-30|      2| 42|
|    11|Karen|Engineering|  NULL|     F|2023-01-05|      3| 24|
|    12|  Leo|    Finance| 78000|     M|2021-11-15|      4| 31|
+------+-----+-----------+------+------+

In [5]:
# SELECT specific columns with WHERE filter
spark.sql("""
    SELECT name, department, salary
    FROM   employees
    WHERE  department = 'Engineering'
""").show()


+-----+-----------+------+
| name| department|salary|
+-----+-----------+------+
|Alice|Engineering| 95000|
|  Bob|Engineering| 88000|
|Grace|Engineering|102000|
|Karen|Engineering|  NULL|
+-----+-----------+------+



In [14]:
# SELECT average salary of each department, ordered by highest average salary
spark.sql("""
    SELECT department, AVG(salary) as avg_salary
    FROM   employees
    group by department
    ORDER BY AVG(salary) DESC
""").show()

+-----------+-----------------+
| department|       avg_salary|
+-----------+-----------------+
|Engineering|          95000.0|
|    Finance|84666.66666666667|
|  Marketing|71333.33333333333|
|         HR|          60500.0|
+-----------+-----------------+



In [15]:
# SELECT specific columns with WHERE filter with department engineering and ORDER BY salary DESC
spark.sql("""
    SELECT name, department, salary
    FROM   employees
    WHERE  department = 'Engineering'
    ORDER BY salary DESC
""").show()

+-----+-----------+------+
| name| department|salary|
+-----+-----------+------+
|Grace|Engineering|102000|
|Alice|Engineering| 95000|
|  Bob|Engineering| 88000|
|Karen|Engineering|  NULL|
+-----+-----------+------+



In [18]:
# ORDER BY — ascending and descending
spark.sql("""
    SELECT name, salary
    FROM   employees
    ORDER  BY salary DESC
    LIMIT  5
""").show()
spark.sql("""
    SELECT department, COUNT(*) As employee_count
FROM employees
GROUP BY department
""").show()


+-----+------+
| name|salary|
+-----+------+
|Grace|102000|
|Alice| 95000|
|  Ivy| 91000|
|  Bob| 88000|
| Hank| 85000|
+-----+------+

+-----------+--------------+
| department|employee_count|
+-----------+--------------+
|Engineering|             4|
|  Marketing|             3|
|         HR|             2|
|    Finance|             3|
+-----------+--------------+



---
## 🟢 Section 4 — DISTINCT, ALIAS, LIKE, IN, BETWEEN

### Key Concepts:
- `DISTINCT` → removes duplicate rows
- `AS` → column / table alias
- `LIKE` → pattern matching (`%` = wildcard)
- `IN` → match against a list of values
- `BETWEEN` → range check (inclusive)

In [ ]:
# DISTINCT departments
spark.sql("""
    SELECT *
    FROM   employees
    
""").show()


+------+-----+-----------+------+------+----------+-------+---+
|emp_id| name| department|salary|gender| hire_date|dept_id|age|
+------+-----+-----------+------+------+----------+-------+---+
|     1|Alice|Engineering| 95000|     F|2019-03-15|      3| 45|
|     2|  Bob|Engineering| 88000|     M|2020-07-01|      3| 32|
|     3|Carol|  Marketing| 72000|     F|2018-11-20|      2| 38|
|     4|David|  Marketing| 68000|     M|2021-01-10|      2| 29|
|     5|  Eve|         HR| 62000|     F|2017-05-05|      1| 41|
|     6|Frank|         HR| 59000|     M|2022-09-12|      1| 27|
|     7|Grace|Engineering|102000|     F|2016-02-28|      3| 50|
|     8| Hank|    Finance| 85000|     M|2020-04-17|      4| 34|
|     9|  Ivy|    Finance| 91000|     F|2019-08-22|      4| 36|
|    10| Jack|  Marketing| 74000|     M|2018-06-30|      2| 42|
|    11|Karen|Engineering|  NULL|     F|2023-01-05|      3| 24|
|    12|  Leo|    Finance| 78000|     M|2021-11-15|      4| 31|
+------+-----+-----------+------+------+

In [23]:
# DISTINCT departments
spark.sql("""
    SELECT DISTINCT department, name
    FROM   employees
    ORDER By department
    
""").show()

+-----------+-----+
| department| name|
+-----------+-----+
|Engineering|Alice|
|Engineering|  Bob|
|Engineering|Grace|
|Engineering|Karen|
|    Finance| Hank|
|    Finance|  Ivy|
|    Finance|  Leo|
|         HR|Frank|
|         HR|  Eve|
|  Marketing|Carol|
|  Marketing|David|
|  Marketing| Jack|
+-----------+-----+



In [28]:
# Like Pattern Matching Get all distinct departments starting name with E and A
spark.sql("""
    SELECT DISTINCT department
    FROM   employees
    WHERE  department LIKE 'E%' OR department LIKE 'A%' OR department LIKE '%R'
    ORDER  BY department
""").show()

+-----------+
| department|
+-----------+
|Engineering|
|         HR|
+-----------+



In [29]:
# DISTINCT departments
spark.sql("""
    SELECT DISTINCT department
    FROM   employees
    ORDER  BY department
""").show()


+-----------+
| department|
+-----------+
|Engineering|
|    Finance|
|         HR|
|  Marketing|
+-----------+



In [30]:
# ALIAS — rename column output
spark.sql("""
    SELECT name        AS employee_name,
           salary      AS annual_salary,
           salary / 12 AS monthly_salary
    FROM   employees
    WHERE  salary IS NOT NULL
    ORDER  BY annual_salary DESC
""").show()


+-------------+-------------+-----------------+
|employee_name|annual_salary|   monthly_salary|
+-------------+-------------+-----------------+
|        Grace|       102000|           8500.0|
|        Alice|        95000|7916.666666666667|
|          Ivy|        91000|7583.333333333333|
|          Bob|        88000|7333.333333333333|
|         Hank|        85000|7083.333333333333|
|          Leo|        78000|           6500.0|
|         Jack|        74000|6166.666666666667|
|        Carol|        72000|           6000.0|
|        David|        68000|5666.666666666667|
|          Eve|        62000|5166.666666666667|
|        Frank|        59000|4916.666666666667|
+-------------+-------------+-----------------+



In [33]:
# LIKE — pattern matching
spark.sql("""
    SELECT name, department
    FROM   employees
    WHERE  name LIKE 'A%'       -- starts with A
       OR  name LIKE '%e'       -- ends with e
""").show()

# IN — match a list
spark.sql("""
    SELECT name, department, salary
    FROM   employees
    WHERE  department IN ('Engineering', 'Finance')
    ORDER  BY salary DESC
""").show()

# BETWEEN — salary range (inclusive)
spark.sql("""
    SELECT name, salary
    FROM   employees
    WHERE  salary BETWEEN 70000 AND 90000
    ORDER  BY salary
""").show()


+-----+-----------+
| name| department|
+-----+-----------+
|Alice|Engineering|
|  Eve|         HR|
|Grace|Engineering|
+-----+-----------+

+-----+-----------+------+
| name| department|salary|
+-----+-----------+------+
|Grace|Engineering|102000|
|Alice|Engineering| 95000|
|  Ivy|    Finance| 91000|
|  Bob|Engineering| 88000|
| Hank|    Finance| 85000|
|  Leo|    Finance| 78000|
|Karen|Engineering|  NULL|
+-----+-----------+------+

+-----+------+
| name|salary|
+-----+------+
|Carol| 72000|
| Jack| 74000|
|  Leo| 78000|
| Hank| 85000|
|  Bob| 88000|
+-----+------+



---
## 🟡 Section 5 — Aggregations: COUNT, SUM, AVG, MIN, MAX + GROUP BY + HAVING

### Key Concepts:
| Function | Purpose |
|----------|---------|
| `COUNT(*)` | Total rows |
| `COUNT(col)` | Rows where col is NOT NULL |
| `SUM(col)` | Total sum |
| `AVG(col)` | Average value |
| `MIN / MAX` | Smallest / Largest |
| `GROUP BY` | Group rows before aggregating |
| `HAVING` | Filter **after** aggregation (like WHERE for groups) |

> ⚠️ `WHERE` filters rows **before** aggregation. `HAVING` filters **after**.

In [34]:
# COUNT, SUM, AVG, MIN, MAX — company-wide stats
spark.sql("""
    SELECT COUNT(*)          AS total_employees,
           COUNT(salary)     AS employees_with_salary,
           SUM(salary)       AS total_salary_bill,
           ROUND(AVG(salary), 2) AS avg_salary,
           MIN(salary)       AS lowest_salary,
           MAX(salary)       AS highest_salary
    FROM   employees
""").show()


+---------------+---------------------+-----------------+----------+-------------+--------------+
|total_employees|employees_with_salary|total_salary_bill|avg_salary|lowest_salary|highest_salary|
+---------------+---------------------+-----------------+----------+-------------+--------------+
|             12|                   11|           874000|  79454.55|        59000|        102000|
+---------------+---------------------+-----------------+----------+-------------+--------------+



In [35]:
# GROUP BY — stats per department
spark.sql("""
    SELECT department,
           COUNT(*)               AS headcount,
           ROUND(AVG(salary), 0)  AS avg_salary,
           MAX(salary)            AS max_salary,
           MIN(salary)            AS min_salary
    FROM   employees
    GROUP  BY department
    ORDER  BY avg_salary DESC
""").show()


+-----------+---------+----------+----------+----------+
| department|headcount|avg_salary|max_salary|min_salary|
+-----------+---------+----------+----------+----------+
|Engineering|        4|   95000.0|    102000|     88000|
|    Finance|        3|   84667.0|     91000|     78000|
|  Marketing|        3|   71333.0|     74000|     68000|
|         HR|        2|   60500.0|     62000|     59000|
+-----------+---------+----------+----------+----------+



In [37]:
# HAVING — keep only departments with avg salary > 75,000
spark.sql("""
    SELECT department,
           COUNT(*)               AS headcount,
           ROUND(AVG(salary), 0)  AS avg_salary
    FROM   employees
    GROUP  BY department
    HAVING AVG(salary) > 75000
    ORDER  BY avg_salary DESC
""").show()

# HAVING — keep only departments with max salary > 5000
spark.sql("""
    SELECT department, avg(salary) AS avg_salary
    FROM   employees
    GROUP  BY department
    HAVING AVG(salary) > 5000
    ORDER  BY avg_salary DESC
""").show()


+-----------+---------+----------+
| department|headcount|avg_salary|
+-----------+---------+----------+
|Engineering|        4|   95000.0|
|    Finance|        3|   84667.0|
+-----------+---------+----------+

+-----------+-----------------+
| department|       avg_salary|
+-----------+-----------------+
|Engineering|          95000.0|
|    Finance|84666.66666666667|
|  Marketing|71333.33333333333|
|         HR|          60500.0|
+-----------+-----------------+



In [42]:
# GROUP BY multiple columns + HAVING with COUNT
spark.sql("""
    SELECT department, gender,
           COUNT(*)              AS total_count,
           ROUND(AVG(salary), 0) AS avg_salary
    FROM   employees
    GROUP  BY department, gender
   -- HAVING COUNT(*) >= 1
    ORDER  BY department, gender
""").show()


+-----------+------+-----------+----------+
| department|gender|total_count|avg_salary|
+-----------+------+-----------+----------+
|Engineering|     F|          3|   98500.0|
|Engineering|     M|          1|   88000.0|
|    Finance|     F|          1|   91000.0|
|    Finance|     M|          2|   81500.0|
|         HR|     F|          1|   62000.0|
|         HR|     M|          1|   59000.0|
|  Marketing|     F|          1|   72000.0|
|  Marketing|     M|          2|   71000.0|
+-----------+------+-----------+----------+



---
## 🟡 Section 6 — JOINs: INNER, LEFT, RIGHT, FULL OUTER

### Visual Guide:
```
INNER JOIN          LEFT JOIN            RIGHT JOIN         FULL OUTER JOIN
   A ∩ B              A + (A∩B)          B + (A∩B)           A ∪ B
  ┌──┐                ┌────┐              ┌────┐           ┌────────┐
  │  │                │ A  │              │    │           │ A    B │
  │██│                │ ██ │              │ ██ │           │ ██ ██ │
  └──┘                └────┘              └────┘           └────────┘
```
| Join Type | Returns |
|-----------|---------|
| `INNER JOIN` | Only matching rows from both tables |
| `LEFT JOIN` | All rows from LEFT + matching from RIGHT (NULL if no match) |
| `RIGHT JOIN` | All rows from RIGHT + matching from LEFT (NULL if no match) |
| `FULL OUTER JOIN` | All rows from both — NULL where no match |

In [ ]:
## Inner Join employees with departments
spark.sql("""
    SELECT name, gender, dept_name, location
           
    FROM   employees, departments
    where employees.dept_id = departments.dept_id
""").show()

spark.sql("""
    SELECT name, gender, dept_name, location
           
    FROM   employees e 
    Inner Join departments d 
   ON e.dept_id = d.dept_id
""").show()

+-----+------+-----------+-------------+
| name|gender|  dept_name|     location|
+-----+------+-----------+-------------+
|  Eve|     F|         HR|     New York|
|Frank|     M|         HR|     New York|
|Carol|     F|  Marketing|      Chicago|
|David|     M|  Marketing|      Chicago|
| Jack|     M|  Marketing|      Chicago|
|Alice|     F|Engineering|San Francisco|
|  Bob|     M|Engineering|San Francisco|
|Grace|     F|Engineering|San Francisco|
|Karen|     F|Engineering|San Francisco|
| Hank|     M|    Finance|     New York|
|  Ivy|     F|    Finance|     New York|
|  Leo|     M|    Finance|     New York|
+-----+------+-----------+-------------+

+-----+------+-----------+-------------+
| name|gender|  dept_name|     location|
+-----+------+-----------+-------------+
|  Eve|     F|         HR|     New York|
|Frank|     M|         HR|     New York|
|Carol|     F|  Marketing|      Chicago|
|David|     M|  Marketing|      Chicago|
| Jack|     M|  Marketing|      Chicago|
|Alice|     F|E

In [51]:
# LEFT Join employees with departments
spark.sql("""
    SELECT name, gender, dept_name, location
           
    FROM   employees e 
    LEFT Join departments d 
   ON e.dept_id = d.dept_id
   ORDER BY name, dept_name
""").show()

+-----+------+-----------+-------------+
| name|gender|  dept_name|     location|
+-----+------+-----------+-------------+
|Alice|     F|Engineering|San Francisco|
|  Bob|     M|Engineering|San Francisco|
|Carol|     F|  Marketing|      Chicago|
|David|     M|  Marketing|      Chicago|
|  Eve|     F|         HR|     New York|
|Frank|     M|         HR|     New York|
|Grace|     F|Engineering|San Francisco|
| Hank|     M|    Finance|     New York|
|  Ivy|     F|    Finance|     New York|
| Jack|     M|  Marketing|      Chicago|
|Karen|     F|Engineering|San Francisco|
|  Leo|     M|    Finance|     New York|
+-----+------+-----------+-------------+



In [49]:
# Right Join with employees and departments
spark.sql("""
    SELECT name, gender, dept_name, location
           
    FROM   employees e 
    Right Join departments d 
   ON e.dept_id = d.dept_id
""").show()

+-----+------+-----------+-------------+
| name|gender|  dept_name|     location|
+-----+------+-----------+-------------+
|Frank|     M|         HR|     New York|
|  Eve|     F|         HR|     New York|
| Jack|     M|  Marketing|      Chicago|
|David|     M|  Marketing|      Chicago|
|Carol|     F|  Marketing|      Chicago|
|Karen|     F|Engineering|San Francisco|
|Grace|     F|Engineering|San Francisco|
|  Bob|     M|Engineering|San Francisco|
|Alice|     F|Engineering|San Francisco|
|  Leo|     M|    Finance|     New York|
|  Ivy|     F|    Finance|     New York|
| Hank|     M|    Finance|     New York|
| NULL|  NULL|      Legal|       Boston|
+-----+------+-----------+-------------+



In [52]:
# INNER JOIN — employees with their department details (only matched rows)
spark.sql("""
    SELECT e.emp_id, e.name, e.salary,
           d.dept_name, d.location, d.budget
    FROM   employees e
    INNER  JOIN departments d ON e.dept_id = d.dept_id
    ORDER  BY e.salary DESC
""").show()


+------+-----+------+-----------+-------------+-------+
|emp_id| name|salary|  dept_name|     location| budget|
+------+-----+------+-----------+-------------+-------+
|     7|Grace|102000|Engineering|San Francisco|2000000|
|     1|Alice| 95000|Engineering|San Francisco|2000000|
|     9|  Ivy| 91000|    Finance|     New York|1200000|
|     2|  Bob| 88000|Engineering|San Francisco|2000000|
|     8| Hank| 85000|    Finance|     New York|1200000|
|    12|  Leo| 78000|    Finance|     New York|1200000|
|    10| Jack| 74000|  Marketing|      Chicago| 800000|
|     3|Carol| 72000|  Marketing|      Chicago| 800000|
|     4|David| 68000|  Marketing|      Chicago| 800000|
|     5|  Eve| 62000|         HR|     New York| 500000|
|     6|Frank| 59000|         HR|     New York| 500000|
|    11|Karen|  NULL|Engineering|San Francisco|2000000|
+------+-----+------+-----------+-------------+-------+



In [53]:
# LEFT JOIN — all departments + their employees (Legal has no employees → NULLs)
spark.sql("""
    SELECT d.dept_name, d.location,
           e.name AS employee_name, e.salary
    FROM   departments d
    LEFT   JOIN employees e ON d.dept_id = e.dept_id
    ORDER  BY d.dept_name, e.salary DESC
""").show()


+-----------+-------------+-------------+------+
|  dept_name|     location|employee_name|salary|
+-----------+-------------+-------------+------+
|Engineering|San Francisco|        Grace|102000|
|Engineering|San Francisco|        Alice| 95000|
|Engineering|San Francisco|          Bob| 88000|
|Engineering|San Francisco|        Karen|  NULL|
|    Finance|     New York|          Ivy| 91000|
|    Finance|     New York|         Hank| 85000|
|    Finance|     New York|          Leo| 78000|
|         HR|     New York|          Eve| 62000|
|         HR|     New York|        Frank| 59000|
|      Legal|       Boston|         NULL|  NULL|
|  Marketing|      Chicago|         Jack| 74000|
|  Marketing|      Chicago|        Carol| 72000|
|  Marketing|      Chicago|        David| 68000|
+-----------+-------------+-------------+------+



In [59]:
# Find departments that have NO employees (using LEFT JOIN + IS NULL trick)
spark.sql("""
    SELECT d.dept_name, d.location
    FROM   departments d
    LEFT   JOIN employees e ON d.dept_id = e.dept_id
    WHERE  e.emp_id IS NULL
""").show()

# FULL OUTER JOIN — all employees + all departments (unmatched on both sides)
spark.sql("""
    SELECT e.name AS employee_name,
           d.dept_name
    FROM   employees e
    FULL   OUTER JOIN departments d ON e.dept_id = d.dept_id
    ORDER  BY d.dept_name
""").show()


+---------+--------+
|dept_name|location|
+---------+--------+
|    Legal|  Boston|
+---------+--------+

+-------------+-----------+
|employee_name|  dept_name|
+-------------+-----------+
|        Alice|Engineering|
|          Bob|Engineering|
|        Grace|Engineering|
|        Karen|Engineering|
|         Hank|    Finance|
|          Ivy|    Finance|
|          Leo|    Finance|
|          Eve|         HR|
|        Frank|         HR|
|         NULL|      Legal|
|        Carol|  Marketing|
|        David|  Marketing|
|         Jack|  Marketing|
+-------------+-----------+



In [60]:
# 3-TABLE JOIN — employees + departments + transactions
spark.sql("""
    SELECT e.name, d.dept_name, t.amount, t.category, t.txn_date
    FROM   employees    e
    JOIN   departments  d ON e.dept_id   = d.dept_id
    JOIN   transactions t ON e.emp_id    = t.emp_id
    ORDER  BY t.amount DESC
""").show()


+-----+-----------+------+-----------+----------+
| name|  dept_name|amount|   category|  txn_date|
+-----+-----------+------+-----------+----------+
|Grace|Engineering|5500.0|      Cloud|2026-01-15|
|Grace|Engineering|4200.0|      Cloud|2026-02-05|
|  Ivy|    Finance|3800.0|    Finance|2026-02-18|
|  Bob|Engineering|3200.0|   Software|2026-01-08|
|Alice|Engineering|2500.0|Electronics|2026-01-05|
| Hank|    Finance|2100.0|    Finance|2026-01-22|
|Alice|Engineering|1800.0|Electronics|2026-01-20|
| Jack|  Marketing|1600.0|  Marketing|2026-01-11|
|Carol|  Marketing|1100.0|  Marketing|2026-02-14|
|Carol|  Marketing| 950.0|  Marketing|2026-02-01|
|David|  Marketing| 750.0|  Marketing|2026-01-30|
|  Eve|         HR| 400.0|   HR Tools|2026-02-10|
+-----+-----------+------+-----------+----------+



---
## 🔴 Section 7 — Subqueries & CTEs (WITH clause)

### Subquery Types:
| Type | Where Used | Example Use |
|------|-----------|-------------|
| **Scalar** | SELECT clause | `(SELECT MAX(salary) FROM employees)` |
| **Inline View** | FROM clause | `FROM (SELECT ... ) AS sub` |
| **Correlated** | WHERE clause | References outer query column |

### CTE (Common Table Expression):
```sql
WITH cte_name AS (
    SELECT ...
)
SELECT * FROM cte_name;
```
> CTEs make complex queries **readable and reusable** within the same query.

In [ ]:
# Scalar Subquery — employees earning above the company average
spark.sql("""
    SELECT name, department, salary,
           (SELECT ROUND(AVG(salary), 0) FROM employees) AS company_avg
    FROM   employees
    WHERE  salary > (SELECT AVG(salary) FROM employees)
    ORDER  BY salary DESC
""").show()


In [ ]:
# Inline View Subquery — top earner per department
spark.sql("""
    SELECT department, name, salary
    FROM (
        SELECT department, name, salary,
               ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) AS rn
        FROM   employees
        WHERE  salary IS NOT NULL
    ) sub
    WHERE rn = 1
    ORDER BY salary DESC
""").show()


In [ ]:
# CTE — department salary stats, then filter high-budget depts
spark.sql("""
    WITH dept_stats AS (
        SELECT e.department,
               COUNT(*)               AS headcount,
               ROUND(AVG(e.salary),0) AS avg_salary,
               SUM(e.salary)          AS total_salary,
               d.budget
        FROM   employees e
        JOIN   departments d ON e.dept_id = d.dept_id
        GROUP  BY e.department, d.budget
    )
    SELECT department, headcount, avg_salary, total_salary,
           budget,
           ROUND(total_salary / budget * 100, 2) AS salary_to_budget_pct
    FROM   dept_stats
    ORDER  BY salary_to_budget_pct DESC
""").show()


In [ ]:
# Multiple CTEs — chain them together
spark.sql("""
    WITH high_earners AS (
        SELECT emp_id, name, salary, department
        FROM   employees
        WHERE  salary > 80000
    ),
    their_transactions AS (
        SELECT h.name, h.department, h.salary,
               COUNT(t.txn_id)    AS txn_count,
               SUM(t.amount)      AS total_spend
        FROM   high_earners h
        LEFT   JOIN transactions t ON h.emp_id = t.emp_id
        GROUP  BY h.name, h.department, h.salary
    )
    SELECT name, department, salary, txn_count, total_spend
    FROM   their_transactions
    ORDER  BY total_spend DESC
""").show()


---
## 🔴 Section 8 — Window Functions: ROW_NUMBER vs RANK vs DENSE_RANK

### The Big Three — Ranking Functions:

```
Salary Data:  [100k, 95k, 95k, 88k, 88k, 72k]

ROW_NUMBER:   [  1,    2,   3,   4,   5,   6 ]  ← always unique, no gaps
RANK:         [  1,    2,   2,   4,   4,   6 ]  ← ties get same rank, GAP after
DENSE_RANK:   [  1,    2,   2,   3,   3,   4 ]  ← ties get same rank, NO gap
```

| Function | Ties | Gaps | Use Case |
|----------|------|------|----------|
| `ROW_NUMBER()` | Different number for ties | No gaps | Pagination, deduplication |
| `RANK()` | Same rank for ties | Yes — skips next rank | Olympic-style rankings |
| `DENSE_RANK()` | Same rank for ties | No gaps | Percentile-style rankings |

### Syntax:
```sql
FUNCTION() OVER (
    PARTITION BY column   -- reset rank for each group
    ORDER BY column DESC  -- determines ranking order  
)
```

In [ ]:
# Side-by-side comparison: ROW_NUMBER vs RANK vs DENSE_RANK
# across ALL employees ranked by salary
spark.sql("""
    SELECT name, department, salary,
           ROW_NUMBER() OVER (ORDER BY salary DESC)          AS row_number,
           RANK()       OVER (ORDER BY salary DESC)          AS rank,
           DENSE_RANK() OVER (ORDER BY salary DESC)          AS dense_rank
    FROM   employees
    WHERE  salary IS NOT NULL
    ORDER  BY salary DESC
""").show()


In [ ]:
# PARTITION BY department — rank resets for each department
spark.sql("""
    SELECT name, department, salary,
           ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) AS row_num,
           RANK()       OVER (PARTITION BY department ORDER BY salary DESC) AS rank,
           DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS dense_rank
    FROM   employees
    WHERE  salary IS NOT NULL
    ORDER  BY department, salary DESC
""").show()


In [ ]:
# Classic interview problem: Top 2 earners per department using ROW_NUMBER
spark.sql("""
    WITH ranked AS (
        SELECT name, department, salary,
               ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) AS rn
        FROM   employees
        WHERE  salary IS NOT NULL
    )
    SELECT department, name, salary, rn
    FROM   ranked
    WHERE  rn <= 2
    ORDER  BY department, rn
""").show()


In [ ]:
# DENSE_RANK use case: Find employees with the 2nd highest salary (handles ties)
spark.sql("""
    WITH salary_ranks AS (
        SELECT name, department, salary,
               DENSE_RANK() OVER (ORDER BY salary DESC) AS dr
        FROM   employees
        WHERE  salary IS NOT NULL
    )
    SELECT name, department, salary
    FROM   salary_ranks
    WHERE  dr = 2
""").show()


---
## 🔴 Section 9 — Window Functions: LAG, LEAD, NTILE, Running Totals

| Function | Description |
|----------|-------------|
| `LAG(col, n)` | Value from **n rows before** current row |
| `LEAD(col, n)` | Value from **n rows after** current row |
| `NTILE(n)` | Splits rows into **n equal buckets** (1 to n) |
| `SUM() OVER (...)` | **Running/cumulative total** |
| `AVG() OVER (...)` | **Rolling average** |
| `FIRST_VALUE()` | First value in the window partition |
| `LAST_VALUE()` | Last value in the window partition |

### Syntax reminder:
```sql
LAG(column, 1, default_value) OVER (PARTITION BY col ORDER BY col)
                   ↑ offset     ↑ value if NULL (optional)
```

In [ ]:
# LAG — compare each transaction amount to the previous one (same employee)
spark.sql("""
    SELECT emp_id, txn_date, amount,
           LAG(amount, 1, 0) OVER (PARTITION BY emp_id ORDER BY txn_date) AS prev_amount,
           amount - LAG(amount, 1, 0) OVER (PARTITION BY emp_id ORDER BY txn_date) AS change
    FROM   transactions
    ORDER  BY emp_id, txn_date
""").show()


In [ ]:
# LEAD — peek at the NEXT transaction date for each employee
spark.sql("""
    SELECT emp_id, txn_date, amount,
           LEAD(txn_date, 1) OVER (PARTITION BY emp_id ORDER BY txn_date) AS next_txn_date,
           LEAD(amount,   1) OVER (PARTITION BY emp_id ORDER BY txn_date) AS next_amount
    FROM   transactions
    ORDER  BY emp_id, txn_date
""").show()


In [ ]:
# Running Total (Cumulative SUM) — transactions ordered by date
spark.sql("""
    SELECT txn_id, txn_date, amount,
           SUM(amount) OVER (ORDER BY txn_date 
                             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
                            ) AS running_total
    FROM   transactions
    ORDER  BY txn_date
""").show()


In [ ]:
# NTILE(4) — split employees into 4 salary quartiles
spark.sql("""
    SELECT name, department, salary,
           NTILE(4) OVER (ORDER BY salary DESC) AS salary_quartile
    FROM   employees
    WHERE  salary IS NOT NULL
    ORDER  BY salary DESC
""").show()


In [ ]:
# FIRST_VALUE / LAST_VALUE — highest and lowest salary in each dept (shown per row)
spark.sql("""
    SELECT name, department, salary,
           FIRST_VALUE(salary) OVER (PARTITION BY department ORDER BY salary DESC) AS dept_max_salary,
           FIRST_VALUE(name)   OVER (PARTITION BY department ORDER BY salary DESC) AS top_earner_in_dept
    FROM   employees
    WHERE  salary IS NOT NULL
    ORDER  BY department, salary DESC
""").show()


---
## 🔴 Section 10 — CASE WHEN, COALESCE, NULL Handling

### Key Concepts:
- `CASE WHEN` → if-else logic inside SQL
- `COALESCE(a, b, c)` → returns **first non-NULL** value
- `NULLIF(a, b)` → returns NULL if a == b, else returns a
- `IS NULL / IS NOT NULL` → NULL checks (`= NULL` does **NOT** work!)
- `IFNULL(col, default)` → Spark shorthand for COALESCE with 2 args

In [ ]:
# CASE WHEN — salary band classification
spark.sql("""
    SELECT name, department, salary,
           CASE
               WHEN salary >= 100000 THEN 'Band A — Executive'
               WHEN salary >= 85000  THEN 'Band B — Senior'
               WHEN salary >= 70000  THEN 'Band C — Mid'
               WHEN salary IS NULL   THEN 'Band Unknown — No Salary'
               ELSE                       'Band D — Junior'
           END AS salary_band
    FROM   employees
    ORDER  BY salary DESC NULLS LAST
""").show(truncate=False)


In [ ]:
# COALESCE — replace NULL salary with dept average
spark.sql("""
    SELECT e.name, e.department,
           e.salary AS original_salary,
           COALESCE(e.salary,
               ROUND(AVG(e2.salary), 0)
           ) AS salary_or_dept_avg
    FROM   employees e
    JOIN (
        SELECT department, AVG(salary) AS salary
        FROM   employees
        WHERE  salary IS NOT NULL
        GROUP  BY department
    ) e2 ON e.department = e2.department
    ORDER  BY e.salary NULLS FIRST
""").show()


In [ ]:
# NULL handling: IS NULL, IS NOT NULL, NULLIF, IFNULL
spark.sql("""
    SELECT name,
           salary,
           IFNULL(salary, -1)             AS salary_default_neg1,
           NULLIF(department, 'HR')       AS dept_no_hr,   -- NULL if HR, else dept name
           salary IS NULL                 AS is_salary_null
    FROM   employees
    ORDER  BY salary NULLS FIRST
""").show()


---
## 🔴 Section 11 — String Functions & Date Functions

### String Functions:
| Function | Example | Result |
|----------|---------|--------|
| `UPPER(s)` | `UPPER('alice')` | `'ALICE'` |
| `LOWER(s)` | `LOWER('ALICE')` | `'alice'` |
| `LENGTH(s)` | `LENGTH('Alice')` | `5` |
| `SUBSTRING(s, pos, len)` | `SUBSTRING('Alice', 1, 3)` | `'Ali'` |
| `CONCAT(a, b)` | `CONCAT('Hello', ' World')` | `'Hello World'` |
| `TRIM(s)` | `TRIM('  hi  ')` | `'hi'` |
| `REPLACE(s, old, new)` | `REPLACE('abc', 'b', 'X')` | `'aXc'` |

### Date Functions:
| Function | Description |
|----------|-------------|
| `CURRENT_DATE` | Today's date |
| `DATEDIFF(end, start)` | Days between two dates |
| `YEAR(date)` | Extract year |
| `MONTH(date)` | Extract month |
| `DATE_ADD(date, n)` | Add n days |
| `DATE_FORMAT(date, fmt)` | Format as string |

In [ ]:
# String functions
spark.sql("""
    SELECT name,
           UPPER(name)                        AS name_upper,
           LOWER(name)                        AS name_lower,
           LENGTH(name)                       AS name_length,
           SUBSTRING(name, 1, 3)              AS first_3_chars,
           CONCAT(name, ' (', department, ')') AS name_with_dept,
           REPLACE(department, 'Engineering', 'Eng') AS dept_short
    FROM   employees
    ORDER  BY name
""").show(truncate=False)


In [ ]:
# Date functions — how long has each employee been at the company?
spark.sql("""
    SELECT name, hire_date,
           YEAR(hire_date)                              AS hire_year,
           MONTH(hire_date)                             AS hire_month,
           DATEDIFF(CURRENT_DATE, hire_date)            AS days_employed,
           ROUND(DATEDIFF(CURRENT_DATE, hire_date)/365, 1) AS years_employed,
           DATE_FORMAT(hire_date, 'MMMM dd, yyyy')      AS formatted_date
    FROM   employees
    ORDER  BY hire_date
""").show(truncate=False)


In [ ]:
# Transactions in year 2026 + month analysis
spark.sql("""
    SELECT YEAR(txn_date)  AS year,
           MONTH(txn_date) AS month,
           COUNT(*)         AS txn_count,
           SUM(amount)      AS total_revenue
    FROM   transactions
    GROUP  BY YEAR(txn_date), MONTH(txn_date)
    ORDER  BY year, month
""").show()


---
## 🏆 Section 12 — Interview-Style Practice Problems

Classic SQL interview questions combining all concepts above.
Try writing the query yourself before expanding the solution cell below each problem.

---
### Problem 1 🟡
**Find the Nth highest salary** (N=3) in the entire company.
> Hint: Use `DENSE_RANK()`

### Problem 2 🟡
**Find employees who earn more than their department average.**
> Hint: Use a subquery or CTE with `AVG()` grouped by department.

### Problem 3 🔴
**For each employee, show their salary rank within their department AND across the whole company.**
> Hint: Two separate `DENSE_RANK()` windows — one with `PARTITION BY`, one without.

### Problem 4 🔴
**Find departments where the total transaction amount is above 5000.**
> Hint: JOIN employees + transactions + departments, GROUP BY, HAVING.

### Problem 5 🔴 (Duplicate Detection)
**Find employees who made more than one transaction (any day).** Show them with their total spend and transaction count.
> Hint: GROUP BY + HAVING COUNT > 1.

In [ ]:
# ✅ Problem 1 Solution — Nth Highest Salary (N=3)
spark.sql("""
    WITH ranked AS (
        SELECT name, salary,
               DENSE_RANK() OVER (ORDER BY salary DESC) AS dr
        FROM   employees
        WHERE  salary IS NOT NULL
    )
    SELECT name, salary, dr AS rank
    FROM   ranked
    WHERE  dr = 3
""").show()


In [ ]:
# ✅ Problem 2 Solution — Employees earning more than their dept average
spark.sql("""
    WITH dept_avg AS (
        SELECT department, ROUND(AVG(salary), 2) AS avg_salary
        FROM   employees
        WHERE  salary IS NOT NULL
        GROUP  BY department
    )
    SELECT e.name, e.department, e.salary,
           da.avg_salary AS dept_avg,
           ROUND(e.salary - da.avg_salary, 2) AS above_avg_by
    FROM   employees e
    JOIN   dept_avg da ON e.department = da.department
    WHERE  e.salary > da.avg_salary
    ORDER  BY above_avg_by DESC
""").show()


In [ ]:
# ✅ Problem 3 Solution — Rank within dept AND across company (dual window)
spark.sql("""
    SELECT name, department, salary,
           DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS dept_rank,
           DENSE_RANK() OVER (ORDER BY salary DESC)                         AS company_rank
    FROM   employees
    WHERE  salary IS NOT NULL
    ORDER  BY department, dept_rank
""").show()


In [ ]:
# ✅ Problem 4 Solution — Departments with total transactions > 5000
spark.sql("""
    SELECT d.dept_name, d.location,
           COUNT(t.txn_id)     AS txn_count,
           SUM(t.amount)       AS total_spend
    FROM   employees    e
    JOIN   departments  d ON e.dept_id = d.dept_id
    JOIN   transactions t ON e.emp_id  = t.emp_id
    GROUP  BY d.dept_name, d.location
    HAVING SUM(t.amount) > 5000
    ORDER  BY total_spend DESC
""").show()


In [ ]:
# ✅ Problem 5 Solution — Employees with multiple transactions + running total
spark.sql("""
    SELECT e.name, e.department,
           COUNT(t.txn_id)  AS txn_count,
           SUM(t.amount)    AS total_spend,
           MIN(t.txn_date)  AS first_txn,
           MAX(t.txn_date)  AS last_txn
    FROM   employees    e
    JOIN   transactions t ON e.emp_id = t.emp_id
    GROUP  BY e.name, e.department
    HAVING COUNT(t.txn_id) > 1
    ORDER  BY total_spend DESC
""").show()


---
## 📋 Quick Reference Cheat Sheet

### Window Function Syntax:
```sql
FUNCTION() OVER (
    [PARTITION BY col1, col2]   -- group/reset boundary
    [ORDER BY col DESC]          -- ordering within the window
    [ROWS BETWEEN ... AND ...]   -- optional frame clause
)
```

### Frame Clause Options:
| Frame | Meaning |
|-------|---------|
| `UNBOUNDED PRECEDING` | From the very first row |
| `CURRENT ROW` | Current row |
| `UNBOUNDED FOLLOWING` | Up to the very last row |
| `1 PRECEDING` | One row before current |
| `1 FOLLOWING` | One row after current |

### When to Use Which Ranking Function:
```
Need unique sequential numbers?        → ROW_NUMBER()
Olympic ranking (gaps after ties)?     → RANK()
Continuous ranking (no gaps)?          → DENSE_RANK()
Split into N equal groups?             → NTILE(N)
Previous row's value?                  → LAG()
Next row's value?                      → LEAD()
```

### Common Interview Patterns:
```sql
-- Top N per group
WHERE rn <= N  (using ROW_NUMBER)

-- Second highest (handles ties)
WHERE dr = 2   (using DENSE_RANK)

-- Employees with no matching record
LEFT JOIN ... WHERE right_table.id IS NULL

-- Running total
SUM(col) OVER (ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
```